# 00 · Setup check: are you ready for *Demo Hero, Production Zero*?

This notebook takes about **1 minute**. It checks five things and ends with a green **✅ You're ready**,
or tells you, in plain English, exactly what to fix.

1. The Python packages install.
2. Your Gemini API key is found.
3. Google accepts the key, and shows which models it can reach.
4. A tiny test question gets an answer.
5. The embedding model (used to search documents in the workshop) works.

**What you need:** a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey),
made in your **own** Google account. Never share your key, and never paste it into a cell.

**Putting the key in Colab (do this once):**
1. Click the 🔑 **key icon** in the left sidebar (*Secrets*).
2. **+ Add new secret**. Name: `GOOGLE_API_KEY`. Value: paste your key.
3. Switch on **Notebook access** for that secret. (Easy to miss. The notebook will tell you if you did.)
4. Menu: **Runtime → Run all**.

**Running on your own laptop instead?** Put `GOOGLE_API_KEY=your-key` in a `.env` file next to this
notebook, or set it as an environment variable.

Cells are labelled 🟢 **RUN**: just run them, you don't need to change anything.

## 🟢 RUN · Step 1: install the packages
Installs Google's official Gemini library (`google-genai`) and a small helper for reading `.env` files.
In production you pin versions like this, so a library update can't break you on the day.
This notebook needs no other files, so there's nothing to download yet.

In [ ]:
import subprocess, sys
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "google-genai>=3.3,<4", "python-dotenv>=1.2,<2"], capture_output=True, text=True)
if _r.returncode == 0:
    print("✅ Packages installed")
else:
    print("❌ Install failed. Check your internet connection, then run this cell again.")
    print(_r.stderr[-800:])

## 🟢 RUN · Step 2: settings (CONFIG)
The model names used in the whole workshop, in one place. We only use models the **free tier** allows:
Flash, Flash-Lite and the Gemini embedding model. The next steps check these against the list of
models your key can actually reach, so if a name ever changes, you'll be told which one to use instead.

In [ ]:
# CONFIG: model names checked against Google's published model list on 2026-11-13.
PREFERRED_MODEL = "gemini-3.8-flash"        # main model for the workshop
FALLBACK_MODEL  = "gemini-3.5-flash-lite"   # used if the main one isn't available to your key
EMBED_MODEL     = "gemini-embedding-001"    # turns text into numbers for searching documents

for _m in (PREFERRED_MODEL, FALLBACK_MODEL, EMBED_MODEL):
    assert "pro" not in _m.lower(), f"{_m} is a Pro model. The free tier can't use it."
print("✅ Settings loaded:", PREFERRED_MODEL, "|", FALLBACK_MODEL, "|", EMBED_MODEL)

## 🟢 RUN · Step 3: find your API key
Looks for the key in **Colab Secrets** first, then an environment variable, then a `.env` file, and
finally asks you to type it (hidden). The key is never printed. In production, keys live in a secret
store, never in code, because code gets shared and keys get leaked.

In [ ]:
import os, getpass

def _find_key():
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # only exists inside Colab
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            key = userdata.get("GOOGLE_API_KEY")
            if key and key.strip():
                return key.strip(), "Colab Secrets"
            print("⚠️  The Colab secret GOOGLE_API_KEY exists but is empty. Paste your key into its Value.")
        except Exception as e:
            kind = type(e).__name__
            if kind == "NotebookAccessError":
                print("⚠️  Found the secret GOOGLE_API_KEY, but this notebook isn't allowed to read it.\n"
                      "    Fix: click the 🔑 key icon in the left sidebar, find GOOGLE_API_KEY and switch\n"
                      "    ON 'Notebook access'. Then: Runtime → Run all.")
            elif kind == "SecretNotFoundError":
                print("⚠️  No Colab secret called GOOGLE_API_KEY.\n"
                      "    Fix: 🔑 key icon in the left sidebar → '+ Add new secret' → Name: GOOGLE_API_KEY\n"
                      "    (exactly, in capitals) → paste your key as the Value → switch on 'Notebook access'.")
            elif kind == "TimeoutException":
                print("⚠️  Colab took too long to hand over the secret. Run this cell again.")
            else:
                print(f"⚠️  Couldn't read Colab Secrets ({kind}).")
    # 2) Environment variable (GOOGLE_API_KEY wins over GEMINI_API_KEY, same as Google's SDK)
    for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        if os.environ.get(name, "").strip():
            return os.environ[name].strip(), f"environment variable {name}"
    # 3) A .env file in this folder or a parent folder
    try:
        from dotenv import dotenv_values, find_dotenv
        path = find_dotenv(usecwd=True)
        if path:
            key = (dotenv_values(path).get("GOOGLE_API_KEY") or "").strip()
            if key:
                return key, ".env file"
    except ImportError:
        pass
    # 4) Ask (hidden input)
    try:
        key = getpass.getpass("Paste your Gemini API key (hidden), then press Enter: ").strip()
        if key:
            return key, "typed in"
    except Exception:
        pass  # no keyboard available (for example, an automated run)
    return None, None

API_KEY, KEY_SOURCE = _find_key()
if API_KEY:
    print(f"✅ Key found ({KEY_SOURCE}). Not printing it, on purpose.")
else:
    print("❌ No API key found.\n"
          "   Fix: get a free key at https://aistudio.google.com/apikey, then either\n"
          "   • Colab: add it as the secret GOOGLE_API_KEY (🔑 icon, switch on 'Notebook access'), or\n"
          "   • Laptop: put GOOGLE_API_KEY=your-key in a .env file next to this notebook.\n"
          "   Then run the notebook again.")

## 🟢 RUN · Step 4: safety net for calling Gemini
Free keys have rate limits that Google doesn't publish. This helper waits and retries when Google says
"slow down" (error 429), and turns every other error into a plain-English fix instead of a wall of red
text. Every production AI app needs this: the model API *will* fail sometimes.

In [ ]:
import time
from google import genai
from google.genai import errors, types

WAITS = [10, 20, 40]          # seconds to wait before each retry
CHECKS = {}                   # what passed, for the summary at the end

class Friendly(Exception):
    "An error we've already explained in plain English."

def explain(e, model=None):
    "Turn a Gemini error into (short reason, fix)."
    code = getattr(e, "code", None)
    msg = str(getattr(e, "message", "") or e)
    low = msg.lower()
    if "api key not valid" in low or "api_key_invalid" in low or code == 401:
        return ("wrong key", "Google says this key isn't valid. Copy it again from "
                "https://aistudio.google.com/apikey (the whole key, no spaces). Make sure it's a "
                "Gemini API key from AI Studio, not another kind of Google key.")
    if "leaked" in low:
        return ("blocked key", "Google blocked this key because it was shared publicly. "
                "Make a new key at https://aistudio.google.com/apikey and keep it secret.")
    if "location" in low and "not supported" in low:
        return ("region", "The Gemini API isn't available in your region. Ask the organisers before the workshop.")
    if code == 404:
        return ("model not available", f"Your key can't use the model '{model}'.")
    if code == 429 and "limit: 0" in low:
        return ("not on free tier", f"The model '{model}' isn't part of your free tier.")
    if code == 429 and "perday" in low.replace(" ", "").replace("_", ""):
        return ("daily limit", "You've used today's free quota for this model. It resets daily; try again tomorrow.")
    if code == 403:
        return ("project not allowed", "Google won't let this key's project use the Gemini API. Usually the fix "
                "is to create a fresh key in a NEW project at https://aistudio.google.com/apikey (some older or "
                "imported Cloud projects need verification).")
    if code == 429:
        return ("too busy", "Free tier says slow down, and it didn't calm down. Wait a minute and run again.")
    if code and code >= 500:
        return ("google busy", "Google's servers are busy right now. Wait a minute and run again.")
    if code is None:
        return ("no connection", "Couldn't reach Google. Check your internet connection (or company proxy/VPN).")
    return ("unexpected", f"Unexpected error {code}: {msg[:200]}")

def call_gemini(fn, model=None):
    "Run fn(); retry politely on 'slow down' errors; raise Friendly with a fix otherwise."
    for attempt in range(len(WAITS) + 1):
        try:
            return fn()
        except Exception as e:
            code = getattr(e, "code", None)
            text = str(e).lower()
            retryable = (code == 429 and "limit: 0" not in text and "perday" not in text.replace("_", "")) \
                        or (isinstance(code, int) and code >= 500)
            if retryable and attempt < len(WAITS):
                print(f"⏳ Free tier says slow down. Waiting {WAITS[attempt]}s…")
                time.sleep(WAITS[attempt])
                continue
            reason, fix = explain(e, model)
            raise Friendly(f"{reason}|{fix}") from None

client = genai.Client(api_key=API_KEY) if API_KEY else None
print("✅ Safety net ready" if client else "⏭  Skipped: no key yet (see Step 3).")

## 🟢 RUN · Step 5: does Google accept the key? Which models can it reach?
Asks Google for the list of models your key can use. We only show the families the workshop uses
(Flash, Flash-Lite, embeddings). In production you check what's really available instead of trusting a
name you remember, because models get renamed and retired.

In [ ]:
AVAILABLE = set()
if client:
    try:
        models = call_gemini(lambda: list(client.models.list()))
        AVAILABLE = {m.name.removeprefix("models/") for m in models}
        CHECKS["key"] = True
        print(f"✅ Google accepted your key. It can see {len(AVAILABLE)} models. The ones we use:\n")
        shown = sorted(n for n in AVAILABLE if ("flash" in n or "embedding" in n) and "pro" not in n
                       and not any(x in n for x in ("image", "tts", "audio", "live", "robotics")))
        for n in shown:
            mark = "⭐ in CONFIG" if n in (PREFERRED_MODEL, FALLBACK_MODEL, EMBED_MODEL) else ""
            print(f"   • {n:40} {mark}")
        for name, label in ((PREFERRED_MODEL, "PREFERRED_MODEL"), (FALLBACK_MODEL, "FALLBACK_MODEL"),
                            (EMBED_MODEL, "EMBED_MODEL")):
            if name not in AVAILABLE:
                print(f"\n⚠️  {label} = '{name}' isn't in your list. Step 6/7 will tell you if that matters.")
    except Friendly as f:
        reason, fix = str(f).split("|", 1)
        CHECKS["key"] = False
        CHECKS["fix"] = fix
        print(f"❌ {fix}")
else:
    print("⏭  Skipped: no key yet (see Step 3).")

## 🟢 RUN · Step 6: one tiny test question
Sends one short question to the main model. If your key can't use it, it automatically falls back to
Flash-Lite and tells you. Temperature 0 means "no creativity": the same question gives the same answer,
which is what you want when testing.

In [ ]:
CHAT_MODEL = None
if CHECKS.get("key"):
    for model in (PREFERRED_MODEL, FALLBACK_MODEL):
        try:
            t0 = time.time()
            r = call_gemini(lambda: client.models.generate_content(
                model=model,
                contents="Reply with exactly one word: ready",
                config=types.GenerateContentConfig(temperature=0, seed=42)), model)
            answer = (r.text or "").strip()
            CHAT_MODEL = model
            print(f"✅ {model} answered “{answer}” in {time.time() - t0:.1f}s")
            if model == FALLBACK_MODEL:
                print(f"   ℹ️  Using Flash-Lite ({FALLBACK_MODEL}) instead of {PREFERRED_MODEL}. "
                      "That's fine for the whole workshop.")
            break
        except Friendly as f:
            reason, fix = str(f).split("|", 1)
            print(f"⚠️  {model}: {fix}")
            if reason not in ("model not available", "not on free tier", "daily limit"):
                CHECKS["fix"] = fix
                break
            if model == PREFERRED_MODEL:
                print(f"   Trying the fallback, {FALLBACK_MODEL}…")
    if CHAT_MODEL is None:
        CHECKS.setdefault("fix", "Neither Flash nor Flash-Lite worked with your key. "
                          "Make a fresh key at https://aistudio.google.com/apikey and run again.")
    CHECKS["chat"] = CHAT_MODEL is not None
else:
    print("⏭  Skipped: the key isn't working yet (see Step 5).")

## 🟢 RUN · Step 7: the embedding model
In the workshop we search a policy document by meaning, not just keywords. That needs the embedding
model, which turns text into a list of numbers. One tiny call checks it works.

In [ ]:
if CHECKS.get("key"):
    try:
        e = call_gemini(lambda: client.models.embed_content(model=EMBED_MODEL, contents="refund for late delivery"),
                        EMBED_MODEL)
        print(f"✅ {EMBED_MODEL} works: one sentence became {len(e.embeddings[0].values)} numbers.")
        CHECKS["embed"] = True
    except Friendly as f:
        reason, fix = str(f).split("|", 1)
        CHECKS["embed"] = False
        CHECKS.setdefault("fix", fix)
        print(f"❌ {EMBED_MODEL}: {fix}")
else:
    print("⏭  Skipped: the key isn't working yet (see Step 5).")

## 🟢 RUN · Result

In [ ]:
ready = bool(CHECKS.get("key") and CHECKS.get("chat") and CHECKS.get("embed"))
line = "=" * 60
if ready:
    print(f"{line}\n✅  YOU'RE READY for Demo Hero, Production Zero!\n{line}")
    print(f"   Key from : {KEY_SOURCE}")
    print(f"   Chat     : {CHAT_MODEL}")
    print(f"   Embedding: {EMBED_MODEL}")
    print("\n   See you at the workshop. Keep this key handy (and secret).")
else:
    print(f"{line}\n❌  NOT READY YET, but it's one fix away\n{line}")
    for label, k in (("Key found", None), ("Key accepted by Google", "key"),
                     ("Test question answered", "chat"), ("Embedding model works", "embed")):
        ok = bool(API_KEY) if k is None else bool(CHECKS.get(k))
        print(f"   {'✅' if ok else '❌'} {label}")
    import sys
    in_colab = "google.colab" in sys.modules
    no_key_fix = ("Get a free key at https://aistudio.google.com/apikey and add it as the Colab secret "
                  "GOOGLE_API_KEY (🔑 icon in the left sidebar, switch on 'Notebook access')." if in_colab else
                  "Get a free key at https://aistudio.google.com/apikey and put GOOGLE_API_KEY=your-key "
                  "in a .env file next to this notebook (or set it as an environment variable).")
    fix = CHECKS.get("fix") or no_key_fix
    print(f"\n👉 What to do: {fix}")
    print("   Then: Runtime → Run all." if in_colab else "   Then run all the cells again.")